# PID control! baby's first PID controller...
Let's start with just proportional control first and see how well that does.

Watched a few videos to get some background; took notes from [this video](https://www.youtube.com/watch?v=XfAt6hNV8XM)
* proportional = if far away, increase input magnitude; gets you to your target as fast as possible
* derivative = prevents overshooting; slow down when closer to target
* integral = removes steady state error (off from target by constant)

Also referenced [this github repo](https://github.com/lukewys/Gadgets-from-Lukewys/tree/master/gym-cartpole-pid) and [this blog post](https://statusfailed.com/blog/2022-12-08-pid-controller-in-the-openai-gym/). I used the gain values from [this file](https://github.com/lukewys/Gadgets-from-Lukewys/blob/master/gym-cartpole-pid/cartpole-PID.py), too lazy to tune my own...

In [41]:
from typing import NamedTuple

class PolePState(NamedTuple):
    """
    P (proportional) controller to keep the pole upright
    """
    goal: float # target pole angle
    k_p: float # P gain

def init_pole_p(k_p=8):
    return PolePState(goal=0., k_p=k_p)

def get_action_pole_p(state, observation, act_key):
    """
    simple PD controller based on the pole angle (which we want to be 0)
    """
    x, x_dot, theta, theta_dot = observation
    error = state.goal - theta
    ctrl_output = state.k_p * error 
    action = 1 if ctrl_output < 0 else 0
    return action

In [42]:
def no_op(state, *args, **kwargs): return state

# state = init_agent()
init_agent = init_pole_p

# state = reset_agent()
reset_agent = init_pole_p # no learning, so reset is the same as init

# action = get_action(state, observation, act_key)
get_action = get_action_pole_p

# state = update(state, observation, reward, terminated, truncated, info)
update = no_op

model_name = "P (proportional) controller"

In [43]:
import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np

env = gym.make(
    "CartPole-v1",
    render_mode="none",
)

N_TRIALS = 100
N_REPS = 10
# TODO load other hyperparams in here somehow?

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    print(f"REPEAT {rep}")

    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    # TODO
    state = init_agent()

    trial_results = []

    for trial_idx in range(N_TRIALS):
        print(f"trial {trial_idx}")

        observation, info = env.reset()

        # TODO optionally reset the agent
        state = reset_agent()

        episode_over = False
        steps = 0

        while not episode_over:
            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # TODO get action (use act_key if it is random)
            action = get_action(state, observation, act_key)

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action)
            steps += 1

            # TODO do something with the new info
            state = update(state, observation, reward, terminated, truncated, info)

            episode_over = terminated or truncated

        trial_results.append(steps)
        print(f"stayed alive for {steps} steps\n")
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
np.save(f"{model_name}_100_trial_results.npy", all_trial_results)

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='none' that is not in the possible render_modes (['human', 'rgb_array']).
  logger.warn(


REPEAT 0
trial 0
stayed alive for 52 steps

trial 1
stayed alive for 38 steps

trial 2
stayed alive for 34 steps

trial 3
stayed alive for 49 steps

trial 4
stayed alive for 54 steps

trial 5
stayed alive for 48 steps

trial 6
stayed alive for 41 steps

trial 7
stayed alive for 25 steps

trial 8
stayed alive for 50 steps

trial 9
stayed alive for 37 steps

trial 10
stayed alive for 50 steps

trial 11
stayed alive for 24 steps

trial 12
stayed alive for 56 steps

trial 13
stayed alive for 39 steps

trial 14
stayed alive for 51 steps

trial 15
stayed alive for 35 steps

trial 16
stayed alive for 27 steps

trial 17
stayed alive for 49 steps

trial 18
stayed alive for 45 steps

trial 19
stayed alive for 36 steps

trial 20
stayed alive for 59 steps

trial 21
stayed alive for 41 steps

trial 22
stayed alive for 40 steps

trial 23
stayed alive for 45 steps

trial 24
stayed alive for 59 steps

trial 25
stayed alive for 34 steps

trial 26
stayed alive for 46 steps

trial 27
stayed alive for 46 

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_trial_results(model_name, mean_curve, std_curve=None):
    """Plot mean steps-until-failure per trial, with optional std band."""
    trials = np.arange(len(mean_curve))
    fig = go.Figure()
    title=f"{model_name} Learning Curve"

    if std_curve is not None:
        # Upper bound (invisible line, anchors the fill)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve + std_curve,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Lower bound (fills down to upper bound)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve - std_curve,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 100, 200, 0.2)",
            name="±1 std",
            hoverinfo="skip",
        ))

    # Mean line on top
    fig.add_trace(go.Scatter(
        x=trials,
        y=mean_curve,
        mode="lines+markers",
        name="Mean steps until failure",
        line=dict(color="rgb(0, 100, 200)"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Trial",
        yaxis_title="Steps until failure",
        template="plotly_white",
    )
    config = {
    'toImageButtonOptions': {
        'format': 'png', # or 'svg', 'jpeg', 'webp'
        'scale': 3 # triple the resolution of the saved img
    }
    }

    fig.show(config=config)

mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)
plot_trial_results(model_name, mean_curve, std_curve)

Unsurprisingly, it does quite poorly. I could tune the gain, but instead, let's add in the D term!

# Now, a PD controller...

In [51]:
from typing import NamedTuple

class PolePDState(NamedTuple):
    """
    PD controller to keep the pole upright
    """
    goal: float # target pole angle
    k_p: float # P gain
    k_d: float # D gain
    last_error: float

def init_pole_pd(k_p=8, k_d=100):
    return PolePDState(goal=0., k_p=k_p, k_d=k_d, last_error=0.)

def get_action_pole_pd(state, observation, act_key):
    """
    simple PD controller based on the pole angle (which we want to be 0)
    """
    x, x_dot, theta, theta_dot = observation
    error = state.goal - theta
    d_error = error - state.last_error
    ctrl_output = state.k_p * error + state.k_d * d_error
    action = 1 if ctrl_output < 0 else 0
    
    state = state._replace(last_error=error) # update state
    return state, action

In [52]:
# we already updated last_error in the get_action step, so no updating needed
# we're not looking at the reward or anything lol
def no_op(state, *args, **kwargs): return state

# state = init_agent()
init_agent = init_pole_pd

# state = reset_agent()
reset_agent = init_pole_pd # no learning, so reset is the same as init

# action = get_action(state, observation, act_key)
get_action = get_action_pole_pd

# state = update(state, observation, reward, terminated, truncated, info)
update = no_op

model_name = "PD controller"

In [53]:
import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np

env = gym.make(
    "CartPole-v1",
    render_mode="none",
)

N_TRIALS = 100
N_REPS = 10
# TODO load other hyperparams in here somehow?

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    print(f"REPEAT {rep}")

    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    # TODO
    state = init_agent()

    trial_results = []

    for trial_idx in range(N_TRIALS):
        print(f"trial {trial_idx}")

        observation, info = env.reset()

        # TODO optionally reset the agent
        state = reset_agent()

        episode_over = False
        steps = 0

        while not episode_over:
            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # TODO get action (use act_key if it is random)
            # NOTE PID updates the state in `get_action` because we need to save `last_error`
            state, action = get_action(state, observation, act_key)

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action)
            steps += 1

            # TODO do something with the new info
            state = update(state, observation, reward, terminated, truncated, info)

            episode_over = terminated or truncated

        trial_results.append(steps)
        print(f"stayed alive for {steps} steps\n")
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
np.save(f"{model_name}_100_trial_results.npy", all_trial_results)

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='none' that is not in the possible render_modes (['human', 'rgb_array']).
  logger.warn(


REPEAT 0
trial 0
stayed alive for 500 steps

trial 1
stayed alive for 500 steps

trial 2
stayed alive for 500 steps

trial 3
stayed alive for 500 steps

trial 4
stayed alive for 500 steps

trial 5
stayed alive for 500 steps

trial 6
stayed alive for 500 steps

trial 7
stayed alive for 500 steps

trial 8
stayed alive for 500 steps

trial 9
stayed alive for 500 steps

trial 10
stayed alive for 500 steps

trial 11
stayed alive for 500 steps

trial 12
stayed alive for 500 steps

trial 13
stayed alive for 500 steps

trial 14
stayed alive for 500 steps

trial 15
stayed alive for 500 steps

trial 16
stayed alive for 500 steps

trial 17
stayed alive for 500 steps

trial 18
stayed alive for 500 steps

trial 19
stayed alive for 500 steps

trial 20
stayed alive for 500 steps

trial 21
stayed alive for 500 steps

trial 22
stayed alive for 500 steps

trial 23
stayed alive for 500 steps

trial 24
stayed alive for 500 steps

trial 25
stayed alive for 500 steps

trial 26
stayed alive for 500 steps

tr

In [54]:
import plotly.graph_objects as go
import numpy as np

def plot_trial_results(model_name, mean_curve, std_curve=None):
    """Plot mean steps-until-failure per trial, with optional std band."""
    trials = np.arange(len(mean_curve))
    fig = go.Figure()
    title=f"{model_name} Learning Curve"

    if std_curve is not None:
        # Upper bound (invisible line, anchors the fill)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve + std_curve,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Lower bound (fills down to upper bound)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve - std_curve,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 100, 200, 0.2)",
            name="±1 std",
            hoverinfo="skip",
        ))

    # Mean line on top
    fig.add_trace(go.Scatter(
        x=trials,
        y=mean_curve,
        mode="lines+markers",
        name="Mean steps until failure",
        line=dict(color="rgb(0, 100, 200)"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Trial",
        yaxis_title="Steps until failure",
        template="plotly_white",
    )
    config = {
    'toImageButtonOptions': {
        'format': 'png', # or 'svg', 'jpeg', 'webp'
        'scale': 3 # triple the resolution of the saved img
    }
    }

    fig.show(config=config)

mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)
plot_trial_results(model_name, mean_curve, std_curve)

Wow, turns out that's good enough to solve cartpole immediately... didn't even need to add in a controller for the cart, or the I in PID! crazy times...